# Imputation notebook
Run notebooks in the following order:\
`preprocessing.ipynb -> imputation.ipynb -> dimensionality_reduction.ipynb -> gmm_clustering.ipynb`

In [ ]:
from __future__ import annotations

from pathlib import Path

import numpy as np
import pandas as pd

from utils.imputation_utils import MICEForest
from utils.saving_utils import load_data, save_data

pd.options.mode.copy_on_write = True
pd.options.future.infer_string = True

In [ ]:
# Read in data
data_discov_path = Path("data/preprocessed/data_discov_reduced.pkl")
data_valid_path = Path("data/preprocessed/data_valid_reduced.pkl")

vars_info_path = Path("data/preprocessed/vars_info_reduced.pkl")

data_discov: pd.DataFrame = load_data(data_discov_path)
data_valid: pd.DataFrame = load_data(data_valid_path)
vars_info: pd.DataFrame = load_data(vars_info_path)

# Set outputs folder
outputs_folder = "outputs/imputed"

# Set number of MICE iterations
mice_iterations = 5

In [ ]:
# variable_schema specifies which variables are used to predict any variable. We need to train the imputation for all
# variables so that the kernel is able to impute variables that are not missing in the training data.

all_vars = data_discov.columns
variable_schema = {var1: [var2 for var2 in all_vars if var2 != var1] for var1 in all_vars}

mice_forest_discov = MICEForest(
    mi_number=1,
    mice_iterations=mice_iterations,
    rng=np.random.default_rng(52352),
    verbose=True,
    variable_schema=variable_schema,
)
# all vars that are non-missing in discovery dataset but missing in validation dataset
mice_forest_valid = MICEForest(
    mi_number=1,
    mice_iterations=mice_iterations,
    rng=np.random.default_rng(49378),
    verbose=True,
    variable_schema=variable_schema,
)

# Train imputation kernels for discovery and validation data
mice_forest_discov.fit(data_discov)
mice_forest_valid.fit(data_valid)

In [ ]:
# Impute data using the fitted kernels
data_discov_imputed = mice_forest_discov.transform(data_discov, use_results_from_training=True)
data_valid_imputed = mice_forest_valid.transform(data_valid, use_results_from_training=True)
data_tf_imputed = mice_forest_discov.transform(data_valid)  # Uses RNG, so repeating leads to different result

In [ ]:
# Save models
save_data(mice_forest_discov, Path(outputs_folder, "imputation_kernel_discov.pkl"))
save_data(mice_forest_valid, Path(outputs_folder, "imputation_kernel_valid.pkl"))

# Save imputed data
save_data(data_discov_imputed, Path(outputs_folder, "data_discov_imputed.pkl"))
save_data(data_valid_imputed, Path(outputs_folder, "data_valid_imputed.pkl"))
save_data(data_tf_imputed, Path(outputs_folder, "data_tf_imputed.pkl"))